# Lesson 0: Welcome to the Quantum Workbench
## From curiosity to computation — a road map for the course

**What you will see in this notebook:**
- What this course is and who it's for
- Four demos of what quantum mechanics can compute — things you can *see*, *feel*, and relate to
- The software we build toward: PySCF
- How to set up your environment
- A road map of the lessons ahead

> This is **not** a physics lesson. It's an orientation — a quick tour of the
> destination before we start walking.

## What Is This Course?

This is a hands-on course in quantum mechanics for scientists and engineers who
want to go beyond textbook understanding. Every lesson follows the same arc:

$$
\text{Concept} \;\longrightarrow\; \text{Math} \;\longrightarrow\; \text{Code} \;\longrightarrow\; \text{Validation}
$$

We start with the simplest quantum systems — one particle, one dimension — and
build up to predicting real molecular properties from first principles.

The bridge between a mathematical equation and a running program is treated as a
**first-class skill**. We call it the *translation layer*: units, discretization,
dimensional analysis, and sanity-checking are never afterthoughts.

**Who is this for?** Advanced undergraduates and entry-level graduate students in
science and engineering, or anyone with some programming experience (Python) who
wants to develop quantitative quantum mechanics skills — not just the theory, but
the ability to turn equations into numbers and check that those numbers make
physical sense.

**By the end of this course, you will be able to predict molecular properties
from first principles — and understand every step of the calculation.**

---
## What Is PySCF?

The destination of this course is [**PySCF**](https://pyscf.org/) — an
open-source quantum chemistry framework written in Python.

PySCF lets you compute molecular properties — energies, geometries, spectra,
reaction barriers — starting from nothing but the Schrödinger equation and the
positions of the atoms. It covers the full ladder of quantum chemistry methods:

| Method | What it does | Accuracy |
|--------|-------------|----------|
| Hartree–Fock (HF) | Mean-field approximation — electrons feel an average potential | Qualitative |
| Density Functional Theory (DFT) | Approximate exchange–correlation via functionals | Good for most chemistry |
| MP2 | Second-order perturbation theory for electron correlation | Better energetics |
| CCSD / CCSD(T) | Coupled cluster — the "gold standard" of quantum chemistry | High accuracy |
| CASSCF | Multi-configurational SCF for bond breaking and excited states | Essential for tough cases |

Under the hood, PySCF is built on NumPy/SciPy with performance-critical
integrals computed in C (via [libcint](https://github.com/sunqm/libcint)).
It uses **atomic units** internally — the same convention we adopt throughout
this course.

**Why PySCF?**
- Pure Python interface — readable, hackable, no black boxes
- Modular design — methods layer naturally (SCF → post-HF → multi-reference)
- Free and open source (Apache-2.0 license)
- Used in real research across computational chemistry and materials science

> **Reference:** Q. Sun et al., "Recent developments in the PySCF program
> package," *J. Chem. Phys.* **153**, 024109 (2020).
> [DOI: 10.1063/5.0006074](https://doi.org/10.1063/5.0006074)
>
> GitHub: [github.com/pyscf/pyscf](https://github.com/pyscf/pyscf)

We won't use PySCF until later in the course. First, we build our own solvers
from scratch so that when we reach PySCF, we understand what it's doing. But
let's peek at what's possible.

---
## Demo A: The Chemical Bond

**What holds a molecule together?** Let's compute it.

We take the simplest molecule — H$_2$, two hydrogen atoms — and ask: how does
the total energy change as we pull the atoms apart? The result is a *potential
energy surface*, and the depth of its minimum is the bond strength.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, cc

# ====== H₂ bond dissociation curve ======
# Scan the H-H distance and compute the total energy at each point.
# The minimum of this curve gives the equilibrium bond length; the depth
# of the well gives the bond dissociation energy.

distances = np.linspace(0.5, 4.0, 30)  # bond lengths in Angstrom
energies_hf = []
energies_ccsd = []

for d in distances:
    # gto.M: build a molecular object (geometry + basis set).
    # 'sto-3g' is a minimal basis set — small and fast, not very accurate.
    mol = gto.M(atom=f'H 0 0 0; H 0 0 {d}', basis='sto-3g',
                unit='Angstrom', verbose=0)

    # Hartree-Fock: each electron sees the average field of all others.
    # Fast but misses electron correlation (instantaneous electron-electron interaction).
    mf = scf.RHF(mol).run()
    energies_hf.append(mf.e_tot)     # total electronic energy in hartree

    # CCSD (Coupled Cluster Singles and Doubles): includes correlation.
    # Much more expensive but systematically improvable.
    mycc = cc.CCSD(mf).run()
    energies_ccsd.append(mycc.e_tot)  # total CCSD energy in hartree

energies_hf = np.array(energies_hf)
energies_ccsd = np.array(energies_ccsd)

# Print equilibrium geometry and dissociation energy for each method
print(f"{'Method':<8} {'R_eq (A)':>10} {'E_eq (Eh)':>12} {'E_dissoc (Eh)':>14}")
print("-" * 48)
for label, E in [("HF", energies_hf), ("CCSD", energies_ccsd)]:
    i_min = np.argmin(E)  # index of the energy minimum
    # Dissociation energy = E(infinity) - E(equilibrium)
    print(f"{label:<8} {distances[i_min]:10.3f} {E[i_min]:12.6f} {E[-1] - E[i_min]:14.6f}")

In [ ]:
# ====== Plot: H₂ potential energy surface ======
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(distances, energies_hf, 'o-', color='#2196F3', markersize=4,
        label='Hartree-Fock')
ax.plot(distances, energies_ccsd, 's-', color='#E91E63', markersize=4,
        label='CCSD')

# Mark equilibrium (energy minimum) for each method
for E, color, marker in [(energies_hf, '#2196F3', 'o'),
                          (energies_ccsd, '#E91E63', 's')]:
    i_min = np.argmin(E)
    ax.plot(distances[i_min], E[i_min], marker, color=color,
            markersize=10, zorder=5)

ax.set_xlabel('Bond distance (Angstrom)', fontsize=12)
ax.set_ylabel('Total energy (Hartree)', fontsize=12)
ax.set_title('H$_2$ potential energy surface', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0.4, 4.1)

# Annotate key features of the curve
i_min_ccsd = np.argmin(energies_ccsd)
ax.annotate('equilibrium\nbond length',
            xy=(distances[i_min_ccsd], energies_ccsd[i_min_ccsd]),
            xytext=(1.8, -0.85), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='gray'),
            ha='center')

# At large R, HF fails because it can't describe bond breaking correctly
ax.annotate('HF and CCSD\ndiverge here',
            xy=(2.5, (energies_hf[20] + energies_ccsd[20]) / 2),
            xytext=(3.2, -1.0), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='gray'),
            ha='center')

plt.tight_layout()
plt.savefig('figures/h2_bond_energy.png', dpi=150, bbox_inches='tight')
plt.show()

This curve is why hydrogen is a stable molecule at room temperature. The
depth of the well is the bond strength — the energy you'd need to supply to
break the molecule apart.

Notice that **HF and CCSD agree near equilibrium but diverge at large
distances**. Hartree-Fock treats each electron as moving in the average field of
all others — a good approximation when the bond is intact, but qualitatively
wrong when it breaks. CCSD includes *electron correlation* and handles
dissociation better.

> *We'll understand exactly why in Lessons 6-8, when we build the Hartree-Fock
> method from scratch and then layer correlation on top.*

---
## Demo B: Where Do Electrons Live?

Electrons don't orbit the nucleus like planets around the sun. Quantum mechanics
gives us **orbitals** — probability clouds that describe where an electron is
most likely to be found.

Let's compute the orbitals of water (H$_2$O) and look at them.

In [ ]:
# ====== Demo B: Molecular orbitals of water ======
from pyscf import gto, scf
from pyscf.tools import cubegen

# Water molecule (HF/cc-pVDZ optimized geometry, in Angstrom)
mol = gto.M(
    atom='''
    O   0.000000   0.000000   0.112032
    H   0.000000   0.748790  -0.466566
    H   0.000000  -0.748790  -0.466566
    ''',
    basis='cc-pvdz',  # correlation-consistent double-zeta basis — a good balance
    unit='Angstrom',
    verbose=0
)

# Run Hartree-Fock to get the molecular orbitals
mf = scf.RHF(mol).run()
nocc = mol.nelectron // 2  # number of doubly-occupied orbitals (5 for water)

print(f"H2O Hartree-Fock energy: {mf.e_tot:.6f} Eh")
print(f"Occupied orbitals: {nocc},  Virtual orbitals: {mf.mo_coeff.shape[1] - nocc}")
# HOMO = highest occupied MO (the frontier orbital that donates electrons)
# LUMO = lowest unoccupied MO (the frontier orbital that accepts electrons)
print(f"HOMO energy: {mf.mo_energy[nocc-1]:.4f} Eh  (orbital {nocc})")
print(f"LUMO energy: {mf.mo_energy[nocc]:.4f} Eh  (orbital {nocc+1})")

# Generate cube files: 3D volumetric data of the orbital on a grid.
# cubegen.orbital evaluates ψ(r) = Σ_μ C_μi χ_μ(r) on an 80³ grid,
# where C_μi are the MO coefficients and χ_μ are the basis functions.
cubegen.orbital(mol, 'figures/h2o_homo.cube', mf.mo_coeff[:, nocc-1],
                nx=80, ny=80, nz=80)
cubegen.orbital(mol, 'figures/h2o_lumo.cube', mf.mo_coeff[:, nocc],
                nx=80, ny=80, nz=80)
print("\nCube files written to figures/")

In [ ]:
import py3Dmol

def show_orbital(cube_path, title="", isovalue=0.05):
    """Render an orbital isosurface from a cube file using py3Dmol.
    
    An isosurface shows the boundary where |ψ(r)| = isovalue.
    Blue = positive lobe, red = negative lobe of the wavefunction.
    """
    with open(cube_path) as f:
        cube_data = f.read()

    viewer = py3Dmol.view(width=500, height=400)
    # Show atoms as ball-and-stick model
    viewer.addModel(cube_data, "cube")
    viewer.setStyle({"stick": {"radius": 0.15},
                     "sphere": {"scale": 0.25}})
    # Positive lobe (ψ > 0) in blue
    viewer.addVolumetricData(cube_data, "cube",
                            {"isoval": isovalue, "color": "#1565C0",
                             "opacity": 0.6})
    # Negative lobe (ψ < 0) in red
    viewer.addVolumetricData(cube_data, "cube",
                            {"isoval": -isovalue, "color": "#E53935",
                             "opacity": 0.6})
    viewer.zoomTo()
    print(title)
    return viewer.show()

show_orbital("figures/h2o_homo.cube",
             "HOMO -- highest occupied molecular orbital (lone-pair character)")

In [ ]:
show_orbital("figures/h2o_lumo.cube",
             "LUMO -- lowest unoccupied molecular orbital (antibonding character)")

In [ ]:
# ====== Static fallback: 2D contour slices of HOMO and LUMO ======
# If py3Dmol doesn't render in your environment, these 2D slices show
# the essential orbital shapes. We evaluate the MO wavefunction on a
# 2D grid by calling PySCF's eval_gto (evaluate Gaussian-type orbitals).
import matplotlib.pyplot as plt
import numpy as np

BOHR_PER_ANGSTROM = 1.8897259886
ngrid = 150

# --- HOMO: slice through xz-plane (y=0) ---
# The HOMO (1b₁ symmetry) has lobes perpendicular to the molecular plane,
# so we need the xz-plane to see them.
x = np.linspace(-3.0, 3.0, ngrid)
z = np.linspace(-3.0, 3.0, ngrid)
X_h, Z_h = np.meshgrid(x, z)
# Build a (ngrid², 3) array of 3D coordinates — all with y=0
coords_homo = np.column_stack([
    X_h.ravel() * BOHR_PER_ANGSTROM,  # PySCF wants coordinates in bohr
    np.zeros(ngrid * ngrid),            # y = 0 (the slice plane)
    Z_h.ravel() * BOHR_PER_ANGSTROM
])
# eval_gto: evaluate all basis functions χ_μ(r) at every grid point
# Returns shape (ngrid², nbasis). The MO is ψ_i(r) = Σ_μ C_μi χ_μ(r).
ao_homo = mol.eval_gto('GTOval_sph', coords_homo)
homo_vals = (ao_homo @ mf.mo_coeff[:, nocc-1]).reshape(ngrid, ngrid)

# --- LUMO: slice through yz-plane (x=0) ---
# The LUMO has density in the molecular plane, so yz-plane shows it best.
y = np.linspace(-3.0, 3.0, ngrid)
z2 = np.linspace(-3.0, 3.0, ngrid)
Y_l, Z_l = np.meshgrid(y, z2)
coords_lumo = np.column_stack([
    np.zeros(ngrid * ngrid),            # x = 0
    Y_l.ravel() * BOHR_PER_ANGSTROM,
    Z_l.ravel() * BOHR_PER_ANGSTROM
])
ao_lumo = mol.eval_gto('GTOval_sph', coords_lumo)
lumo_vals = (ao_lumo @ mf.mo_coeff[:, nocc]).reshape(ngrid, ngrid)

# --- Plot: filled contours with the nodal surface (ψ=0) highlighted ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HOMO in xz-plane
ax = axes[0]
vmax = np.max(np.abs(homo_vals)) * 0.6  # symmetric color scale
c = ax.contourf(X_h, Z_h, homo_vals, levels=30,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax)
# The nodal surface (ψ=0) is where the orbital changes sign
ax.contour(X_h, Z_h, homo_vals, levels=[0], colors='k', linewidths=0.5)
plt.colorbar(c, ax=ax, label='$\\phi(x, z)$', shrink=0.9)
# Mark atom positions projected onto this plane
for lbl, az in [('O', 0.112), ('H', -0.467)]:
    ax.plot(0, az, 'ko', markersize=6)
    ax.annotate(lbl, (0, az), textcoords="offset points",
                xytext=(5, 5), fontsize=10, fontweight='bold')
ax.set_xlabel('x (Angstrom)', fontsize=11)
ax.set_ylabel('z (Angstrom)', fontsize=11)
ax.set_title('HOMO  (xz-plane, y = 0)', fontsize=13)
ax.set_aspect('equal')

# LUMO in yz-plane
ax = axes[1]
vmax = np.max(np.abs(lumo_vals)) * 0.6
c = ax.contourf(Y_l, Z_l, lumo_vals, levels=30,
                cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.contour(Y_l, Z_l, lumo_vals, levels=[0], colors='k', linewidths=0.5)
plt.colorbar(c, ax=ax, label='$\\phi(y, z)$', shrink=0.9)
atom_yz = [(0.0, 0.112), (0.749, -0.467), (-0.749, -0.467)]
for (ay, az), lbl in zip(atom_yz, ['O', 'H', 'H']):
    ax.plot(ay, az, 'ko', markersize=6)
    ax.annotate(lbl, (ay, az), textcoords="offset points",
                xytext=(5, 5), fontsize=10, fontweight='bold')
ax.set_xlabel('y (Angstrom)', fontsize=11)
ax.set_ylabel('z (Angstrom)', fontsize=11)
ax.set_title('LUMO  (yz-plane, x = 0)', fontsize=13)
ax.set_aspect('equal')

fig.suptitle('H$_2$O molecular orbitals — 2D slices',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/h2o_orbitals_2d.png', dpi=150, bbox_inches='tight')
plt.show()

The **HOMO** (highest occupied molecular orbital) of water has 1b₁ symmetry —
its lobes point perpendicular to the molecular plane. That's why we sliced
through the xz-plane to see them. This out-of-plane electron density is the
**lone pair** that makes water such a good hydrogen-bond donor and solvent.

The **LUMO** (lowest unoccupied molecular orbital) is antibonding — it has a
**node** (a surface where the wavefunction is zero) between the oxygen and
hydrogen atoms. If you could promote an electron into this orbital, it would
weaken the O–H bonds.

> *These shapes come from the same eigenvalue equation we solve in Lesson 1 —
> just in 3D, with a more complicated Hamiltonian and a basis of Gaussian
> functions instead of grid points.*

---
## Demo C: Why Molecules Vibrate

Your microwave oven works because water molecules vibrate and rotate at specific
quantum-mechanical frequencies. Those frequencies aren't arbitrary — they're
determined by the shape of the potential energy surface near the equilibrium
geometry.

If we know the curvature of the energy surface (the **Hessian** — the matrix of
second derivatives of the energy with respect to atomic positions), we can
predict exactly which frequencies of infrared light a molecule absorbs.

In [ ]:
# ====== Demo C: Vibrational frequencies of water ======
# The key idea: vibrational frequencies come from the curvature of the
# potential energy surface at the equilibrium geometry, just like a
# spring constant determines the frequency of a mass on a spring.
import numpy as np
from pyscf import gto, scf

# Water at HF/cc-pVDZ optimized geometry
mol = gto.M(
    atom='''
    O   0.000000   0.000000   0.112032
    H   0.000000   0.748790  -0.466566
    H   0.000000  -0.748790  -0.466566
    ''',
    basis='cc-pvdz',
    unit='Angstrom',
    verbose=0
)
mf = scf.RHF(mol).run()

# ====== Step 1: Compute the Hessian (matrix of second derivatives) ======
# The Hessian H_ij = ∂²E/∂R_i∂R_j tells us the curvature of the energy
# surface. For N atoms, this is a (3N × 3N) matrix (x, y, z per atom).
hess_obj = mf.Hessian().run()
hessian = hess_obj.de  # shape: (natm, natm, 3, 3) — PySCF's block format

# Reshape from (natm, natm, 3, 3) blocks into a flat (3N × 3N) matrix
natm = mol.natm
n = 3 * natm  # 9 for water (3 atoms × 3 coordinates)
H = np.zeros((n, n))
for i in range(natm):
    for j in range(natm):
        H[3*i:3*i+3, 3*j:3*j+3] = hessian[i, j]

# ====== Step 2: Mass-weight the Hessian ======
# The vibrational equation is M·a = -H·x, or equivalently:
#   M^{-1/2} H M^{-1/2} · q = ω² · q
# where q = M^{1/2} · x are mass-weighted coordinates.
# This transforms the coupled Newton's equations into a standard eigenvalue problem.
masses_amu = mol.atom_mass_list()          # atomic masses in amu
AMU_TO_ME = 1822.888486209                  # 1 amu in electron masses (a.u.)
M_au = np.repeat(masses_amu * AMU_TO_ME, 3) # repeat each mass 3× (x,y,z)
Mhalf_inv = 1.0 / np.sqrt(M_au)
# np.einsum: efficient element-wise scaling of rows and columns
# H_mw[i,j] = (1/√M_i) · H[i,j] · (1/√M_j)
H_mw = np.einsum('i,ij,j->ij', Mhalf_inv, H, Mhalf_inv)

# ====== Step 3: Diagonalize → normal mode frequencies ======
# Eigenvalues = ω² (in atomic units), eigenvectors = normal mode displacements.
eigenvalues, eigenvectors = np.linalg.eigh(H_mw)

# Convert eigenvalues to frequencies in cm⁻¹ (spectroscopic units).
# ω (a.u.) × 219474.63 = ω (cm⁻¹).
# np.sign handles the 6 near-zero eigenvalues (translations/rotations)
# that may be slightly negative due to numerical noise.
AU_TO_CM1 = 219474.63
freqs_cm1 = np.sqrt(np.abs(eigenvalues)) * np.sign(eigenvalues) * AU_TO_CM1

# Separate translations/rotations (~0 cm⁻¹) from real vibrations (>> 100 cm⁻¹)
vib_mask = np.abs(freqs_cm1) > 100
vib_freqs = freqs_cm1[vib_mask]       # 3 vibrational frequencies for water
vib_modes = eigenvectors[:, vib_mask]  # corresponding displacement patterns

# Compare to experimental fundamentals (which include anharmonicity)
expt = [1595, 3657, 3756]
mode_names = ['bend (scissors)', 'symmetric stretch', 'asymmetric stretch']

print(f"{'Mode':<25} {'Computed (cm-1)':>16} {'Experiment (cm-1)':>18}")
print("-" * 62)
for i, (name, freq, exp) in enumerate(zip(mode_names, vib_freqs, expt)):
    print(f"{name:<25} {freq:16.1f} {exp:18d}")
print()
print("Note: HF overestimates harmonic frequencies by ~10-15%.")
print("The experimental values include anharmonicity and correlation effects.")

In [ ]:
# ====== Visualize normal mode displacement patterns ======
# Each normal mode is a collective motion of all atoms. The eigenvectors
# from the Hessian diagonalization give the displacement direction of each
# atom in mass-weighted coordinates. We un-mass-weight to get physical displacements.
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Atom positions (Angstrom) for drawing the molecule
atom_pos = np.array([
    [0.000,  0.000,  0.112],   # O
    [0.000,  0.749, -0.467],   # H
    [0.000, -0.749, -0.467],   # H
])
atom_labels = ['O', 'H', 'H']
atom_colors = ['#E53935', '#BBBBBB', '#BBBBBB']  # red for O, gray for H
atom_sizes = [200, 120, 120]

mode_names_short = ['Bending\n(scissors)', 'Symmetric\nstretch', 'Asymmetric\nstretch']

for idx, ax in enumerate(axes):
    # Get the displacement pattern for this normal mode
    q = vib_modes[:, idx].reshape(natm, 3)
    # Un-mass-weight: physical displacement = M^{-1/2} × mass-weighted displacement
    for a in range(natm):
        q[a] /= np.sqrt(masses_amu[a] * AMU_TO_ME)

    # Normalize displacements for display (arrows scale to 0.35 Angstrom)
    q = q / np.max(np.abs(q)) * 0.35

    # Draw atoms as colored circles
    for a in range(natm):
        y, z = atom_pos[a, 1], atom_pos[a, 2]
        ax.scatter(y, z, s=atom_sizes[a], c=atom_colors[a],
                   edgecolors='black', linewidths=1, zorder=3)
        ax.annotate(atom_labels[a], (y, z), textcoords="offset points",
                    xytext=(8, 8), fontsize=10, fontweight='bold')

        # Draw displacement arrows (blue) showing how each atom moves
        dy, dz = q[a, 1], q[a, 2]
        if np.sqrt(dy**2 + dz**2) > 0.01:
            ax.annotate('', xy=(y + dy, z + dz), xytext=(y, z),
                        arrowprops=dict(arrowstyle='->', color='#1565C0',
                                        lw=2.5))

    # Draw O-H bonds
    for i, j in [(0, 1), (0, 2)]:
        ax.plot([atom_pos[i, 1], atom_pos[j, 1]],
                [atom_pos[i, 2], atom_pos[j, 2]],
                'k-', linewidth=1.5, zorder=1)

    ax.set_title(f'{mode_names_short[idx]}\n{vib_freqs[idx]:.0f} cm$^{{-1}}$',
                 fontsize=11)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.0, 0.8)
    ax.set_aspect('equal')
    ax.axis('off')

fig.suptitle('Normal modes of H$_2$O', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/h2o_normal_modes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Schematic IR absorption spectrum ======
# In a real experiment, you'd shine infrared light through a sample and
# measure which frequencies are absorbed. Here we simulate that by placing
# Lorentzian peaks at each computed vibrational frequency.
fig, ax = plt.subplots(figsize=(10, 3.5))

# Lorentzian lineshape: I(ν) = 1 / (1 + ((ν - ν₀)/γ)²)
# where ν₀ is the peak center and γ is the half-width.
wavenumbers = np.linspace(500, 4500, 2000)
spectrum = np.zeros_like(wavenumbers)
width = 40  # cm⁻¹ broadening (artificial, for display)

for freq in vib_freqs:
    spectrum += 1.0 / (1.0 + ((wavenumbers - freq) / width) ** 2)

ax.fill_between(wavenumbers, spectrum, alpha=0.3, color='#E53935')
ax.plot(wavenumbers, spectrum, color='#E53935', linewidth=1.5)

# Annotate peaks — offset labels to avoid overlap
offsets = [(-60, 0.35), (-250, 0.20), (80, 0.35)]
for (freq, name), (dx, dy) in zip(
        zip(vib_freqs, ['bend', 'sym. stretch', 'asym. stretch']), offsets):
    ax.annotate(f'{name}\n{freq:.0f} cm$^{{-1}}$',
                xy=(freq, 1.02), xytext=(freq + dx, 1.02 + dy),
                ha='center', fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Wavenumber (cm$^{-1}$)', fontsize=12)
ax.set_ylabel('Absorption (a.u.)', fontsize=12)
ax.set_title('Computed IR absorption spectrum of H$_2$O', fontsize=13)
ax.set_xlim(500, 4500)
ax.set_ylim(0, 1.7)

plt.tight_layout()
plt.savefig('figures/h2o_ir_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

These frequencies determine which infrared light water absorbs. The
bending mode near 1600 cm$^{-1}$ and the stretching modes near 3700 cm$^{-1}$ are
signatures that spectroscopists use to identify water in any sample — from
a glass of tap water to the atmosphere of an exoplanet.

This is also why **water vapor is a greenhouse gas**: it absorbs infrared
radiation (heat) at exactly these wavelengths. The greenhouse effect isn't
magic — it's quantum mechanics.

> *The connection between force constants (curvature of the energy surface)
> and vibrational frequencies is the same physics as a mass on a spring —
> we'll build this from first principles in Lesson 2 when we solve the
> quantum harmonic oscillator.*

---
## Demo D: Why Things Are Colored

The color of a substance comes from quantum mechanics. When a molecule absorbs
light, it jumps from its ground electronic state to an excited state. The
**energy gap** between these states determines the wavelength of light absorbed.

Let's compute the electronic excitations of formaldehyde (H$_2$CO) — a small
molecule with a well-known absorption in the ultraviolet.

In [ ]:
# ====== Demo D: Electronic excitations of formaldehyde ======
# When a molecule absorbs light, an electron jumps from an occupied orbital
# to an unoccupied one. The energy gap determines the wavelength of light.
from pyscf import gto, dft, tdscf

# Formaldehyde (H₂CO) — a small molecule with a well-known UV absorption
mol_hcho = gto.M(
    atom='''
    C   0.0000   0.0000   0.0000
    O   0.0000   0.0000   1.2030
    H   0.0000   0.9370  -0.5870
    H   0.0000  -0.9370  -0.5870
    ''',
    basis='cc-pvdz',
    unit='Angstrom',
    verbose=0
)

# DFT with B3LYP functional — a widely used exchange-correlation approximation.
# DFT is often better than HF for excited-state energies because it includes
# some electron correlation through the functional.
mf_hcho = dft.RKS(mol_hcho)
mf_hcho.xc = 'b3lyp'
mf_hcho.kernel()

# TDDFT (Time-Dependent DFT): computes excited-state energies and transition
# properties (oscillator strengths) from the ground-state DFT solution.
# Each excitation is a linear combination of occupied→virtual orbital transitions.
td = tdscf.TDDFT(mf_hcho)
td.nstates = 6     # compute the 6 lowest excited states
td.verbose = 0
td.kernel()

# Convert excitation energies to eV and compute oscillator strengths
HARTREE_TO_EV = 27.211386245988
excitation_ev = td.e * HARTREE_TO_EV
# Oscillator strength f: proportional to the absorption intensity.
# f = 0 means the transition is forbidden (symmetry-forbidden or dipole-forbidden).
osc_strength = td.oscillator_strength()

print(f"{'State':<8} {'Energy (eV)':>12} {'Wavelength (nm)':>16} {'Osc. strength':>14}")
print("-" * 54)
for i, (ev, f) in enumerate(zip(excitation_ev, osc_strength)):
    nm = 1239.8 / ev  # E (eV) × λ (nm) = hc = 1239.8 eV·nm
    marker = "  <- n->pi*" if i == 0 else ""
    print(f"  S{i+1:<5} {ev:12.3f} {nm:16.1f} {f:14.4f}{marker}")

In [ ]:
# ====== Plot: electronic absorption spectrum on a wavelength axis ======
# Shows computed transitions as vertical sticks on an electromagnetic
# spectrum background, so you can see which are UV, visible, or IR.
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 4))

# Color-coded background showing regions of the electromagnetic spectrum
spectrum_ranges = [
    (100, 200, '#8B00FF', 'Far UV'),
    (200, 380, '#6A0DAD', 'UV'),
    (380, 450, '#7B00FF', 'Violet'),
    (450, 495, '#0000FF', 'Blue'),
    (495, 570, '#00FF00', 'Green'),
    (570, 590, '#FFFF00', 'Yellow'),
    (590, 620, '#FF8C00', 'Orange'),
    (620, 750, '#FF0000', 'Red'),
    (750, 1000, '#8B0000', 'Near IR'),
]

for wl_min, wl_max, color, label in spectrum_ranges:
    ax.axvspan(wl_min, wl_max, alpha=0.15, color=color)
    mid = (wl_min + wl_max) / 2
    ax.text(mid, 1.35, label, ha='center', fontsize=7, color='gray')

# Visible range boundaries
ax.axvline(380, color='gray', linestyle=':', linewidth=0.8)
ax.axvline(750, color='gray', linestyle=':', linewidth=0.8)

# Plot each computed transition as a vertical stick at its wavelength
# Height = oscillator strength (proportional to absorption intensity)
for i, (ev, f) in enumerate(zip(excitation_ev, osc_strength)):
    nm = 1239.8 / ev
    if nm > 100:
        # Color bright transitions red, weak ones gray
        color = '#E53935' if f > 0.01 else '#999999'
        ax.vlines(nm, 0, max(f, 0.005), color=color, linewidth=2.5)
        if f > 0.001 or i == 0:
            label_text = f'S{i+1}: {nm:.0f} nm'
            if i == 0:
                label_text += r' (n$\rightarrow\pi$*)'
            ax.annotate(label_text, xy=(nm, max(f, 0.005)),
                        xytext=(nm + 30, f + 0.08), fontsize=9,
                        arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Oscillator strength', fontsize=12)
ax.set_title('Computed electronic spectrum of formaldehyde (H$_2$CO) -- TDDFT/B3LYP',
             fontsize=13)
ax.set_xlim(100, 800)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.savefig('figures/formaldehyde_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

The lowest excitation of formaldehyde — the famous **n$\rightarrow\pi$\*
transition** — falls around 300 nm in the ultraviolet. It has near-zero
oscillator strength because it is symmetry-forbidden (the transition dipole
moment vanishes by symmetry). That's why formaldehyde is colorless: it doesn't
absorb visible light.

**Color = energy gaps between quantum states.** Bigger conjugated molecules
(think dyes, pigments, chlorophyll) have smaller HOMO-LUMO gaps, pushing their
absorption into the visible range. That's why carrots are orange ($\beta$-carotene),
leaves are green (chlorophyll), and the sky is blue (Rayleigh scattering, but
that's a different story).

> *We'll compute excited states ourselves in Lessons 9-10, once we have the
> DFT and TDDFT machinery in place.*

---
## Beyond Toy Demos — Where This Goes in Practice

The demos above used small molecules and modest basis sets — things that run in
seconds on a laptop. But PySCF scales to problems that matter outside the
classroom. Here's a taste.

### Real chemistry at real scale

**Drug design** requires understanding a molecule's conformational energy
landscape — how the energy changes as you rotate bonds. GPU-accelerated PySCF
([GPU4PySCF](https://github.com/pyscf/gpu4pyscf)) has been used to run torsion
scans of drug-sized molecules like enzalutamide (a prostate cancer treatment):
over 1,000 DFT evaluations with large basis sets, completed in under 10 hours.

We can demonstrate the idea on a much smaller molecule. Hydrogen peroxide
(H$_2$O$_2$) has a single rotatable bond — the H–O–O–H dihedral. Let's scan it
and compute the energy at every angle:

In [ ]:
# ====== Demo E: Torsion scan of H₂O₂ ======
# Rotate the H-O-O-H dihedral angle and compute the energy at each point.
# This is the same idea behind drug conformational analysis — understanding
# how energy changes as you rotate bonds.
import numpy as np
from pyscf import gto, scf

# Fixed internal coordinates (experimental values)
r_OO = 1.47    # O-O bond length (Angstrom)
r_OH = 0.97    # O-H bond length (Angstrom)
angle = np.radians(94.8)  # H-O-O bond angle

# Scan the dihedral from 0° (cis) to 360° in 10° steps
dihedrals = np.arange(0, 361, 10)
energies_torsion = []

for phi_deg in dihedrals:
    phi = np.radians(phi_deg)

    # Build Cartesian coordinates from internal coordinates.
    # O1 at origin, O2 along z-axis. H1 in the xz-plane.
    # H2 rotated by the dihedral angle φ around the O-O axis.
    O1 = [0.0, 0.0, 0.0]
    O2 = [0.0, 0.0, r_OO]
    H1 = [r_OH * np.sin(angle), 0.0, -r_OH * np.cos(angle)]
    H2 = [r_OH * np.sin(angle) * np.cos(phi),
           r_OH * np.sin(angle) * np.sin(phi),
           r_OO + r_OH * np.cos(angle)]

    mol = gto.M(
        atom=f'O {O1[0]:.6f} {O1[1]:.6f} {O1[2]:.6f}; '
             f'O {O2[0]:.6f} {O2[1]:.6f} {O2[2]:.6f}; '
             f'H {H1[0]:.6f} {H1[1]:.6f} {H1[2]:.6f}; '
             f'H {H2[0]:.6f} {H2[1]:.6f} {H2[2]:.6f}',
        basis='cc-pvdz',
        unit='Angstrom',
        verbose=0
    )
    mf = scf.RHF(mol).run()
    energies_torsion.append(mf.e_tot)

energies_torsion = np.array(energies_torsion)

# Convert to relative energies in kcal/mol (chemist's units for barriers)
HARTREE_TO_KCALMOL = 627.509474
rel_energies = (energies_torsion - energies_torsion.min()) * HARTREE_TO_KCALMOL

# Print key conformations
print(f"{'Dihedral (°)':<14} {'Rel. energy (kcal/mol)':>22}")
print("-" * 38)
for deg, E in zip(dihedrals, rel_energies):
    if deg % 60 == 0:
        label = {0: '(cis)', 180: '(trans)', 360: '(cis)'}.get(deg, '')
        print(f"{deg:<14d} {E:22.2f}  {label}")

In [ ]:
# ====== Plot: H₂O₂ torsion energy profile ======
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(dihedrals, rel_energies, 'o-', color='#2196F3', markersize=4)

# Mark key conformations
i_min = np.argmin(rel_energies)
i_cis = 0
i_trans = np.argmin(np.abs(dihedrals - 180))

ax.annotate(f'cis (eclipsed)\n{rel_energies[i_cis]:.1f} kcal/mol',
            xy=(0, rel_energies[i_cis]),
            xytext=(50, rel_energies[i_cis] - 1.0),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate(f'trans\n{rel_energies[i_trans]:.1f} kcal/mol',
            xy=(180, rel_energies[i_trans]),
            xytext=(220, rel_energies[i_trans] + 1.5),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate(f'equilibrium ({dihedrals[i_min]}°)',
            xy=(dihedrals[i_min], rel_energies[i_min]),
            xytext=(dihedrals[i_min] + 30, -0.8),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('H–O–O–H dihedral angle (degrees)', fontsize=12)
ax.set_ylabel('Relative energy (kcal/mol)', fontsize=12)
ax.set_title('H$_2$O$_2$ torsion scan — conformational energy landscape',
             fontsize=13)
ax.set_xlim(-5, 365)
ax.set_xticks(np.arange(0, 361, 60))

plt.tight_layout()
plt.savefig('figures/h2o2_torsion.png', dpi=150, bbox_inches='tight')
plt.show()

The cis conformation (0°) has the highest energy — the two hydrogen atoms are
eclipsed, and their lone pairs repel. The equilibrium sits near 110–120°, and
there's a secondary barrier at the trans conformation (180°). This is the kind
of energy profile that determines molecular shape, and for drug molecules with
many rotatable bonds, mapping these landscapes is essential for predicting
binding affinity.

**Battery chemistry.** Researchers use PySCF to study surface reactions at
lithium-ion battery electrode interfaces — understanding why batteries degrade,
what side reactions occur during charging, and how electrolyte molecules interact
with electrode surfaces. The same eigenvalue machinery we'll build in this
course underlies those calculations.

**Predicting spectra before synthesis.** PySCF can compute NMR chemical shifts,
IR frequencies (Demo C), UV-Vis absorption (Demo D), and more — all from first
principles. You could predict what the NMR spectrum of a candidate molecule
looks like *before* anyone synthesizes it.

### It's a Python library, not a black box

This is the single most important thing about PySCF that distinguishes it from
traditional quantum chemistry packages. Programs like Gaussian or ORCA use
domain-specific input files — you write a text file, submit it, and get results
back. PySCF is different: it's a Python library. You `import pyscf` the same way
you `import numpy`.

That means you can embed quantum chemistry calculations inside optimization
loops, machine learning pipelines, or automated screening workflows. The torsion
scan above is a simple example — we just put a PySCF calculation inside a `for`
loop. Here's another: computing a molecular property across a series of
molecules in a few lines of code:

In [ ]:
# ====== Demo F: Molecular screening — HOMO-LUMO gaps in a loop ======
# Six molecules, one loop, one property. This is what "PySCF is a Python
# library" looks like in practice.
import numpy as np
from pyscf import gto, scf
import matplotlib.pyplot as plt

molecules = {
    'H₂':  'H 0 0 0; H 0 0 0.74',
    'HF':  'H 0 0 0; F 0 0 0.92',
    'CO':  'C 0 0 0; O 0 0 1.13',
    'H₂O': 'O 0 0 0; H 0 0.76 0.59; H 0 -0.76 0.59',
    'NH₃': 'N 0 0 0; H 0 0.94 -0.31; H 0.81 -0.47 -0.31; H -0.81 -0.47 -0.31',
    'CH₄': 'C 0 0 0; H 0.63 0.63 0.63; H -0.63 -0.63 0.63; '
           'H -0.63 0.63 -0.63; H 0.63 -0.63 -0.63',
}

HARTREE_TO_EV = 27.211386245988
names = []
gaps = []

print(f"{'Molecule':<10} {'HOMO (eV)':>10} {'LUMO (eV)':>10} {'Gap (eV)':>10}")
print("-" * 44)

for name, geom in molecules.items():
    mol = gto.M(atom=geom, basis='cc-pvdz', unit='Angstrom', verbose=0)
    mf = scf.RHF(mol).run()
    nocc = mol.nelectron // 2
    homo = mf.mo_energy[nocc - 1] * HARTREE_TO_EV
    lumo = mf.mo_energy[nocc] * HARTREE_TO_EV
    gap = lumo - homo
    names.append(name)
    gaps.append(gap)
    print(f"{name:<10} {homo:10.2f} {lumo:10.2f} {gap:10.2f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2196F3', '#E91E63', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
bars = ax.bar(names, gaps, color=colors, edgecolor='black', linewidth=0.5)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{gap:.1f}', ha='center', fontsize=10)

ax.set_ylabel('HOMO–LUMO gap (eV)', fontsize=12)
ax.set_title('HOMO–LUMO gaps across small molecules (HF/cc-pVDZ)', fontsize=13)
ax.set_ylim(0, max(gaps) + 2)

plt.tight_layout()
plt.savefig('figures/homo_lumo_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

The HOMO–LUMO gap is a rough proxy for chemical stability and reactivity — large
gaps mean the molecule is hard to excite, small gaps mean it's more reactive.
The point here isn't the chemistry; it's that **we computed a meaningful property
for six molecules in a single code cell**. Replace the gap calculation with
solvation energies, dipole moments, or reaction barriers, and you have the
skeleton of a real screening workflow.

### The bigger picture

Everything in this section — torsion scans, molecular screening, spectral
prediction — is built on the same computational pattern we'll develop throughout
this course:

$$
\text{Define system} \;\longrightarrow\; \text{Build Hamiltonian}
\;\longrightarrow\; \text{Solve eigenvalue problem}
\;\longrightarrow\; \text{Extract properties}
$$

The Hamiltonian gets more complicated (from a 1D grid to Gaussian basis sets,
from one electron to many), and the "solve" step gets more sophisticated (from
direct diagonalization to self-consistent field iterations), but the *pattern*
is the same from Lesson 1 all the way to industrial quantum chemistry.

PySCF's reach extends well beyond what we've shown here. A few directions you
might not expect:

- **Periodic systems.** PySCF handles crystals, surfaces, and materials — not
  just isolated molecules. You can compute band structures of semiconductors or
  study catalytic surfaces with the same tools.
- **Automatic differentiation.** PySCFAD can differentiate *through* entire
  quantum chemistry calculations, enabling inverse design: "what molecular
  geometry gives me the property I want?"
- **Quantum computing.** PySCF is becoming the standard classical backbone for
  hybrid quantum-classical computing experiments, providing the molecular
  integrals and orbital framework that quantum hardware needs.
- **Machine learning.** Researchers use PySCF as a data factory — generating
  high-accuracy quantum chemistry training data for neural network potentials
  and ML models that predict molecular properties at a fraction of the cost.

We won't touch these topics in this course, but the foundation we build —
understanding Hamiltonians, basis representations, eigenvalue problems, and
self-consistency — is the same foundation they all rest on.

In [ ]:
# ====== Demo F: Molecular screening — HOMO-LUMO gaps ======
# Compute the HOMO-LUMO gap for a series of small molecules in a loop.
# This is just Python — you loop over molecules the same way you'd loop
# over anything else. In practice, you might screen hundreds of candidates.
from pyscf import gto, scf

molecules = {
    'H2':  'H 0 0 0; H 0 0 0.74',
    'HF':  'H 0 0 0; F 0 0 0.92',
    'CO':  'C 0 0 0; O 0 0 1.13',
    'H2O': 'O 0 0 0; H 0 0.76 0.59; H 0 -0.76 0.59',
    'NH3': 'N 0 0 0; H 0 0.94 -0.31; H 0.81 -0.47 -0.31; H -0.81 -0.47 -0.31',
    'CH4': 'C 0 0 0; H 0.63 0.63 0.63; H -0.63 -0.63 0.63; '
           'H -0.63 0.63 -0.63; H 0.63 -0.63 -0.63',
}

HARTREE_TO_EV = 27.211386245988
names = []
gaps = []

print(f"{'Molecule':<10} {'HOMO (eV)':>10} {'LUMO (eV)':>10} {'Gap (eV)':>10}")
print("-" * 44)

for name, geom in molecules.items():
    mol = gto.M(atom=geom, basis='cc-pvdz', unit='Angstrom', verbose=0)
    mf = scf.RHF(mol).run()
    nocc = mol.nelectron // 2
    homo = mf.mo_energy[nocc - 1] * HARTREE_TO_EV
    lumo = mf.mo_energy[nocc] * HARTREE_TO_EV
    gap = lumo - homo
    names.append(name)
    gaps.append(gap)
    print(f"{name:<10} {homo:10.2f} {lumo:10.2f} {gap:10.2f}")

---
## Course Road Map

We start with one particle in one dimension and build up to everything you just
saw.

| Lesson | Topic | What you will learn |
|--------|-------|---------------------|
| **00** | **Welcome & Overview** | *(you are here)* |
| 01 | Particle in a 1D box | Schrodinger equation, discretization, eigenvalue problems, units |
| 02 | Harmonic oscillator | New potential, same solver; perturbation theory, ladder operators |
| 03 | 2D/3D systems, hydrogen atom | Separation of variables, angular momentum, radial equation |
| 04 | Variational principle & basis sets | From grids to basis expansions, Gaussian basis functions |
| 05 | Two-electron problem (helium) | Electron-electron repulsion, the need for self-consistency |
| 06 | Hartree-Fock from scratch | SCF procedure, Fock matrix, orbital energies |
| 07 | Hartree-Fock with PySCF | Real molecules, basis set convergence, PySCF workflow |
| 08 | Electron correlation | MP2, CCSD concepts; PySCF correlation methods |
| 09 | DFT fundamentals | Exchange-correlation functionals, Kohn-Sham equations |
| 10+ | Advanced topics | CASSCF, excited states, periodic systems, ... |

Each lesson includes exercises of graduated difficulty — from "vary a parameter
and plot the result" to "implement an extension" to "connect this to a real
physical system.\"

---
## Setting Up Your Environment

This section walks you through setting up a Python environment that will work
for the entire course. We use **conda** (via
[Miniconda](https://docs.conda.io/en/latest/miniconda.html)) to manage
packages, because PySCF has compiled C extensions that conda handles cleanly.

### Step 1: Install Miniconda

If you don't already have conda, install Miniconda (a minimal conda installer):

- **Linux/macOS:** Download from https://docs.conda.io/en/latest/miniconda.html
  and run the installer, or:
  ```bash
  # Linux example
  wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
  bash Miniconda3-latest-Linux-x86_64.sh
  ```
- **Windows:** Download the `.exe` installer from the same page and follow the
  GUI. Use the "Anaconda Prompt" for all commands below.

After installation, restart your terminal and verify:
```bash
conda --version
```

### Step 2: Create a dedicated environment

Isolate this course's packages in their own environment so they don't interfere
with other projects:

```bash
conda create -n quantum python=3.12 -y
conda activate quantum
```

> **Tip:** Always activate the environment (`conda activate quantum`) before
> running notebooks or installing packages. Your terminal prompt should show
> `(quantum)` when the environment is active.

### Step 3: Install packages

```bash
conda install -c conda-forge numpy scipy matplotlib pyscf py3dmol geometric jupyterlab ipykernel -y
```

What each package does:

| Package | Role |
|---------|------|
| `numpy` | Array operations — the backbone of all our numerics |
| `scipy` | Linear algebra solvers (`eigh`), special functions, optimizers |
| `matplotlib` | Plotting — every lesson produces figures |
| `pyscf` | Quantum chemistry framework — the destination of this course |
| `py3dmol` | Interactive 3D molecular visualization in Jupyter |
| `geometric` | Geometry optimizer (used by PySCF for molecular structure optimization) |
| `jupyterlab` | The notebook environment itself |
| `ipykernel` | Lets Jupyter see this conda environment as an available kernel |

### Step 4: Register the Jupyter kernel

This step ensures that Jupyter can find the conda environment, even if you
launch JupyterLab from a different environment or a system install:

```bash
conda activate quantum
python -m ipykernel install --user --name quantum --display-name "Python (quantum)"
```

Then in JupyterLab, select **Kernel > Change Kernel > Python (quantum)**.

> **Common pitfall:** If `import pyscf` fails inside a notebook but works in
> your terminal, the notebook is using the wrong kernel. Check the kernel name
> in the top-right corner of JupyterLab. It should say "Python (quantum)", not
> "Python 3 (ipykernel)" from your base environment.

### Step 5: Launch JupyterLab

```bash
conda activate quantum
cd /path/to/quantum-workbench
jupyter lab
```

This opens JupyterLab in your browser. Navigate to any lesson notebook and run
the cells.

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: No module named 'pyscf'` | Wrong kernel — switch to "Python (quantum)" |
| `ModuleNotFoundError: No module named 'geometric'` | Run `conda install -c conda-forge geometric -y` |
| py3Dmol renders a blank box | Make sure you're in JupyterLab (not a plain terminal); try `jupyter labextension list` |
| Plots don't display inline | Add `%matplotlib inline` at the top of the notebook |
| PySCF prints excessive output | Set `verbose=0` on the `Mole` object or the method object |

### Alternative: pip install (if you prefer not to use conda)

```bash
python -m venv quantum-env
source quantum-env/bin/activate      # Linux/macOS
# quantum-env\Scripts\activate       # Windows

pip install numpy scipy matplotlib pyscf py3Dmol geometric jupyterlab ipykernel
python -m ipykernel install --user --name quantum --display-name "Python (quantum)"
```

Note: the pip route may require a C compiler for PySCF on some platforms. Conda
is more reliable for this reason.

In [ ]:
# ====== Environment verification ======
# Check that all required packages import correctly.
# If anything fails here, see the installation instructions above.
import sys
print(f"Python: {sys.version}")
print()

packages = {
    'numpy': 'np',
    'scipy': 'sp',
    'matplotlib': 'mpl',
    'pyscf': 'pyscf',
    'py3Dmol': 'py3Dmol',
}

all_ok = True
for pkg, alias in packages.items():
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', '(no version attr)')
        print(f"  + {pkg:<14} {version}")
    except ImportError:
        print(f"  x {pkg:<14} NOT FOUND")
        all_ok = False

print()
if all_ok:
    print("All packages available. You're ready to go!")
else:
    print("Some packages are missing. See the installation instructions above.")

---
## What's Next

In **Lesson 1**, we solve the Schrodinger equation for the simplest possible
system: one particle trapped in a box. We'll discretize the equation on a grid,
turn it into a matrix eigenvalue problem, and compute energy levels and
wavefunctions from scratch.

The same equation — with a bigger Hamiltonian, a better basis, and more
electrons — is behind *everything* you just saw in this notebook.

Let's begin.